# Forecasting

With the [time-series fundamentals](time-series-fundamentals.ipynb) in place —
chronological splitting, lag features — we can forecast. Good practice first:
establish a **baseline** so a "real" model has something concrete to beat.

In [ ]:
// A synthetic series like the fundamentals chapter's (trend + weekly season),
// with a little reproducible noise so a forecast has something real to get wrong
// (a perfectly deterministic series is trivially forecastable and hides model
// differences — and collapses ARIMA's prediction intervals to zero width).
let series: Vec<f64> = {
    let mut seed = 12345u64;
    (0..56).map(|t| {
        seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let noise = ((seed >> 11) as f64 / (1u64 << 53) as f64 - 0.5) * 1.4;
        10.0 + 0.2 * t as f64 + 3.0 * ((t as f64) * 2.0 * std::f64::consts::PI / 7.0).sin() + noise
    }).collect()
};
let period = 7usize;
let split = 44usize;
let horizon = series.len() - split;   // forecast the final 12 days

// Seasonal-naive baseline: predict each day using the value one week earlier.
let seasonal_naive: Vec<f64> = (0..horizon)
    .map(|i| series[split - period + (i % period)])
    .collect();
println!("horizon = {} days", horizon);
println!("seasonal-naive forecast starts: {:.2}, {:.2}, {:.2} ...",
         seasonal_naive[0], seasonal_naive[1], seasonal_naive[2]);

## An MSTL model

[`augurs`](https://docs.rs/augurs) provides MSTL (Multiple Seasonal-Trend
decomposition), which models the trend and seasonal structure and forecasts
forward. We fit on the training window and predict the horizon:

In [ ]:
:dep augurs = { version = "0.10", features = ["mstl", "ets"] }
:dep chronos-ts = "0.1"
:dep ndarray = { version = "0.15" }
use augurs::mstl::MSTLModel;
use augurs::prelude::*;

// Fitted-model type isn't nameable across cells, so we return the forecast Vec.
let mstl_forecast: Vec<f64> = {
    let train = &series[..split];
    let model = MSTLModel::naive(vec![period]);
    let fitted = model.fit(train).unwrap();
    fitted.predict(horizon, None).unwrap().point
};
println!("MSTL forecast starts: {:.2}, {:.2}, {:.2} ...",
         mstl_forecast[0], mstl_forecast[1], mstl_forecast[2]);

## Auto-ARIMA with `chronos-ts`

MSTL models the trend/seasonal structure directly. **ARIMA** takes a different
route — it models the series' own autocorrelation (AR), differencing (I), and
moving-average (MA) terms. Picking the orders by hand is fiddly, so
[`chronos-ts`](https://crates.io/crates/chronos-ts) provides **auto-ARIMA**: it
searches `(p,d,q)(P,D,Q)ₘ` orders by information criterion (AICc) and fits the
best one — and, unlike the point forecasts above, returns **prediction
intervals**. We give it the weekly period `m = 7`.

In [ ]:
use chronos_ts::{auto_arima, AutoArimaOptions};
use ndarray::Array1;

// Auto-ARIMA on the training window; keep only the point forecast for the shared
// comparison below, but also show the 95% prediction interval it provides.
let arima_forecast: Vec<f64> = {
    let train = Array1::from_vec(series[..split].to_vec());
    let opts = AutoArimaOptions { m: period, ..Default::default() };
    let model = auto_arima(&train, opts).expect("auto_arima failed");
    let fc = model.forecast_with_intervals(&train, horizon);
    println!("selected model AIC = {:.2}", model.aic());
    println!("day 1 forecast = {:.2}   95% PI = [{:.2}, {:.2}]",
             fc.mean[0], fc.lower_95[0], fc.upper_95[0]);
    fc.mean.to_vec()
};

## Evaluation

Score both forecasts against the held-out actuals with the standard time-series
metrics — **MAE**, **RMSE**, and **MAPE** — computed over the chronological test
window. The model earns its keep only if it beats the baseline:

In [ ]:
{
    let actual = &series[split..];
    let report = |name: &str, pred: &[f64]| {
        let n = actual.len() as f64;
        let mae = actual.iter().zip(pred).map(|(a, p)| (a - p).abs()).sum::<f64>() / n;
        let rmse = (actual.iter().zip(pred).map(|(a, p)| (a - p).powi(2)).sum::<f64>() / n).sqrt();
        let mape = actual.iter().zip(pred).map(|(a, p)| ((a - p) / a).abs()).sum::<f64>() / n * 100.0;
        println!("{:<15} MAE={:.3}  RMSE={:.3}  MAPE={:.1}%", name, mae, rmse, mape);
    };
    report("seasonal-naive", &seasonal_naive);
    report("MSTL", &mstl_forecast);
    report("Auto-ARIMA", &arima_forecast);
}

```{note}
**Ecosystem maturity.** `augurs` is a real, actively maintained time-series
toolkit (forecasting, seasonality/changepoint/outlier detection). But Rust's
time-series ecosystem is younger and narrower than Python's (`statsmodels`,
`prophet`, `sktime`) — verify `augurs`' current API against its docs before
relying on it, since it's actively developed.
```

Next: [Working with larger-than-memory data](../11-larger-than-memory/streaming-and-lazy-execution.ipynb) —
scaling the data pipeline beyond what fits in RAM.